<a href="https://colab.research.google.com/github/rathans48/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule, in plain words: "A page is worth reviewing first if it's old and still getting meaningful search visibility and its click-through rate is unusually low for how well it's positioned." That combination — old, visible, underperforming on clicks relative to position — is the classic "stale but still showing up" archetype pattern from Lane 3's own candidate list.

Reason codes this rule can output (one per row, priority order):

stale_visible_ctr_gap — old, visible, AND CTR well below its position tier's norm
stale_but_visible — old and visible, but CTR is in line with its tier (no separate CTR problem)
not_flagged — doesn't meet the age/visibility bar; score is 0

Action label: review_for_refresh for either flagged reason code, no_action otherwise. I'm deliberately not calling this "fix" or "refresh now" — Lane 3's own framing is decision-support, not a directive.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

monthly = con.sql(f"""
    SELECT
        d.content_hash_id,
        ANY_VALUE(d.client_hash_id) AS client_hash_id,
        SUM(d.gsc_impressions) AS impressions_month,
        SUM(d.gsc_clicks) AS clicks_month,
        AVG(d.gsc_avg_position) AS avg_position,
        DATE_DIFF('day', ANY_VALUE(c.content_created_date), DATE '2026-03-31') AS content_age_days
    FROM {DAILY} d
    JOIN {CONTENT} c ON d.content_hash_id = c.content_hash_id
    GROUP BY d.content_hash_id
""").df()

monthly['ctr'] = monthly['clicks_month'] / monthly['impressions_month'].replace(0, pd.NA)
monthly['age_bucket'] = pd.cut(monthly['content_age_days'],
    bins=[0, 180, 365, 10000], labels=['0-180d', '181-365d', '365d+'])

age_check = monthly.groupby('age_bucket').agg(
    n=('content_hash_id', 'size'),
    avg_impressions=('impressions_month', 'mean')
)
age_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_557/795640349.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_check = monthly.groupby('age_bucket').agg(


,n,avg_impressions
age_bucket,,
0-180d,115644,1183.821677
181-365d,180833,608.118186
365d+,32491,1039.909144


Verdict: MIXED. The naive story — "older pages get less visible" — doesn't hold cleanly here. Impressions actually drop in the middle bucket (181–365d) before rising back up in the oldest bucket (365d+). That's not a monotonic decline, so I can't claim age alone predicts declining visibility. This is exactly the kind of honest negative the skill flags as useful: it tells me content_age_days alone is a weak signal, which is why the rule doesn't rely on age in isolation — it's gated by impressions_month >= 500 too, so a stale-but-still-visible page still needs to clear a real traffic bar to score.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

monthly['position_tier'] = pd.cut(monthly['avg_position'],
    bins=[0, 3, 10, 20, 1000], labels=['top_3', 'page_1', 'page_2_3', 'beyond'])

pos_check = monthly.groupby('position_tier').agg(
    n=('content_hash_id', 'size'),
    avg_ctr=('ctr', 'mean')
)
pos_check

/tmp/ipykernel_557/2063053185.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pos_check = monthly.groupby('position_tier').agg(


,n,avg_ctr
position_tier,,
top_3,16144,0.010589
page_1,81987,0.004926
page_2_3,32204,0.003211
beyond,44969,0.001928


Verdict: CONFIRMED. Clean, monotonic decline — CTR falls as position gets worse, exactly as expected from real search behavior. This is the flag-linked signal the assignment requires, and it holds up, which is why the rule compares each page's CTR against its own tier's median rather than a single global CTR threshold.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

df = monthly.copy()

stale = (df['content_age_days'] >= 365).astype(int)
visible = (df['impressions_month'] >= 500).astype(int)

# tier-relative CTR expectation, from the signal-2 check above
tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
ctr_gap = (df['ctr'] < (tier_median_ctr * 0.5)).astype(int).fillna(0)

df['score'] = stale * visible * df['impressions_month']  # readable on purpose, per the skill

def reason_code(row_stale, row_visible, row_gap):
    if row_stale and row_visible and row_gap:
        return 'stale_visible_ctr_gap'
    elif row_stale and row_visible:
        return 'stale_but_visible'
    else:
        return 'not_flagged'

df['reason_code'] = [reason_code(s, v, g) for s, v, g in zip(stale, visible, ctr_gap)]
df['action_label'] = np.where(df['reason_code'] != 'not_flagged', 'review_for_refresh', 'no_action')

queue = df.sort_values('score', ascending=False)[
    ['content_hash_id', 'client_hash_id', 'score', 'reason_code', 'action_label',
     'impressions_month', 'avg_position', 'ctr', 'content_age_days']
]

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
queue.head(20)

/tmp/ipykernel_557/3042762840.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')


,content_hash_id,client_hash_id,score,reason_code,action_label,impressions_month,avg_position,ctr,content_age_days
219158,content_eadb33b5df496f4a,client_e547b89c05043229,617124.0,stale_but_visible,review_for_refresh,617124.0,2.383011,0.009185,375
220053,content_ec2e0346994fb5a5,client_e547b89c05043229,245276.0,stale_but_visible,review_for_refresh,245276.0,2.854514,0.006034,434
52999,content_0e03de7680314cd5,client_e547b89c05043229,221310.0,stale_but_visible,review_for_refresh,221310.0,2.675217,0.003253,375
52964,content_8d7d99f109e19aa2,client_e547b89c05043229,203497.0,stale_but_visible,review_for_refresh,203497.0,2.563756,0.00142,375
52994,content_4ffe18112a5642e3,client_e547b89c05043229,186983.0,stale_but_visible,review_for_refresh,186983.0,2.331060,0.003134,375
105555,content_471d9cabce329a66,client_73cda7b4e4f265ea,164885.0,stale_but_visible,review_for_refresh,164885.0,4.656030,0.002402,375
166332,content_fd2117c2c6790e4b,client_73cda7b4e4f265ea,151166.0,stale_but_visible,review_for_refresh,151166.0,3.391428,0.002699,410
814,content_e241d6415ac9e534,client_73cda7b4e4f265ea,142304.0,stale_but_visible,review_for_refresh,142304.0,3.276016,0.00241,412
396,content_8e1334d6356668e3,client_73cda7b4e4f265ea,134984.0,stale_but_visible,review_for_refresh,134984.0,4.545582,0.000007,410
166170,content_00d4fdf6e48a2d38,client_73cda7b4e4f265ea,126836.0,stale_but_visible,review_for_refresh,126836.0,5.305905,0.003682,410


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20).copy()

# Back the review: show raw click counts (not just CTR) and compare against tier median
top20 = top20.merge(
    df[['content_hash_id', 'position_tier']], on='content_hash_id', how='left'
)
top20['clicks_month'] = df.set_index('content_hash_id').loc[top20['content_hash_id'], 'clicks_month'].values
top20['tier_median_ctr'] = tier_median_ctr.loc[top20.index] if top20.index.isin(tier_median_ctr.index).all() else \
    df.groupby('position_tier')['ctr'].transform('median').reindex(top20.index)

top20[['content_hash_id', 'client_hash_id', 'impressions_month', 'clicks_month',
       'ctr', 'position_tier', 'avg_position']]

,content_hash_id,client_hash_id,impressions_month,clicks_month,ctr,position_tier,avg_position
0,content_eadb33b5df496f4a,client_e547b89c05043229,617124.0,5668.0,0.009185,top_3,2.383011
1,content_ec2e0346994fb5a5,client_e547b89c05043229,245276.0,1480.0,0.006034,top_3,2.854514
2,content_0e03de7680314cd5,client_e547b89c05043229,221310.0,720.0,0.003253,top_3,2.675217
3,content_8d7d99f109e19aa2,client_e547b89c05043229,203497.0,289.0,0.00142,top_3,2.563756
4,content_4ffe18112a5642e3,client_e547b89c05043229,186983.0,586.0,0.003134,top_3,2.331060
5,content_471d9cabce329a66,client_73cda7b4e4f265ea,164885.0,396.0,0.002402,page_1,4.656030
6,content_fd2117c2c6790e4b,client_73cda7b4e4f265ea,151166.0,408.0,0.002699,page_1,3.391428
7,content_e241d6415ac9e534,client_73cda7b4e4f265ea,142304.0,343.0,0.00241,page_1,3.276016
8,content_8e1334d6356668e3,client_73cda7b4e4f265ea,134984.0,1.0,0.000007,page_1,4.545582
9,content_00d4fdf6e48a2d38,client_73cda7b4e4f265ea,126836.0,467.0,0.003682,page_1,5.305905


All 20 top rows carry `reason_code = stale_but_visible`, not `stale_visible_ctr_gap` — worth noting before the individual reviews: none of the very top pages triggered the CTR-gap flag, even though several (rows 2, 3, 9, 11) have CTRs well under 1%. That's because `score = impressions_month` for any stale+visible page (both multipliers are 0/1), so the ranking is effectively "sort stale+visible pages by raw impression volume" — the CTR-gap logic exists but isn't influencing who reaches the top. That's a real structural weakness, addressed in section 4.

1. `content_eadb33b5df496f4a` (client `...e547b89c05043229`) — 617,124 impressions, position 2.4, CTR 0.92%. Flagged for review; confidence: solid (top_3 position with meaningfully thin CTR for that tier). Would be wrong if this is a branded/navigational query where low CTR is normal (users type the URL instead of clicking the result).

2. `content_ec2e0346994fb5a5` (same client) — 245,276 impressions, position 2.9, CTR 0.60% — well under the top_3 tier average (1.06%). Confidence: solid. Wrong if seasonal dip only, not a lasting pattern (single-month data can't rule this out).

3. `content_0e03de7680314cd5` (same client) — 221,310 impressions, position 2.7, CTR 0.33%. Confidence: solid, real gap vs tier norm. Wrong if a SERP feature (featured snippet, etc.) is siphoning clicks — that's a different fix than a title/meta rewrite.

4. `content_8d7d99f109e19aa2` (same client) — 203,497 impressions, position 2.6, CTR 0.14% — notably thin. Confidence: moderate-high. Wrong if this is a duplicate/near-duplicate of another page on the list cannibalizing its own clicks.

5. `content_4ffe18112a5642e3` (same client) — 186,983 impressions, position 2.3, CTR 0.31%. Confidence: solid. Wrong if intent mismatch (ranking for a query it doesn't actually answer) — rewrite alone wouldn't fix that.

6. `content_471d9cabce329a66` (client `...73cda7b4e4f265ea`) — 164,885 impressions, position 4.7, CTR 0.24%. Confidence: moderate. Wrong if position 4.7 sits at the edge of `page_1` where lower CTR is somewhat expected regardless.

7. `content_fd2117c2c6790e4b` (same client) — 151,166 impressions, position 3.4, CTR 0.27%. Confidence: solid. Wrong if recent SERP layout change unrelated to the page itself.

8. `content_e241d6415ac9e534` (same client) — 142,304 impressions, position 3.3, CTR 0.24%. Confidence: solid. Same caveat as above.

9. `content_8e1334d6356668e3` (same client) — 134,984 impressions, position 4.5, CTR 0.0007% — essentially zero clicks against massive impressions. Confidence: low, not high — this looks more like a tracking/data anomaly than an SEO problem; flagged in weak picks below.

10. `content_00d4fdf6e48a2d38` (same client) — 126,836 impressions, position 5.3, CTR 0.37%. Confidence: moderate.

11. `content_fec55986a1868d62` (same client) — 124,075 impressions, position 9.4, CTR 0.0008% — same near-zero pattern as #9. Confidence: low; same anomaly flag.

12. `content_545bb6cc7081ded3` (client `...e547b89c05043229`) — 122,905 impressions, position 2.6, CTR 0.23%. Confidence: solid.

13. `content_77276ad7a26f4905` (same client) — 116,707 impressions, position 3.9, CTR 0.17%. Confidence: solid.

14. `content_b17c1d1cb0a346d6` (client `...73cda7b4e4f265ea`) — 115,947 impressions, position 4.7, CTR 0.42%. Confidence: moderate.

15. `content_f86f77b3ebdc05ee` (client `...e547b89c05043229`) — 105,420 impressions, position 3.9, CTR 0.52%. Confidence: moderate — closer to typical for its tier, weaker case than others above it.

16. `content_21309e9a83c83653` (same client) — 103,187 impressions, position 5.0, CTR 0.19%. Confidence: solid.

17. `content_cf651123f1085418` (client `...73cda7b4e4f265ea`) — 101,363 impressions, position 6.3, CTR 0.17%. Confidence: moderate.

18. `content_e73024da2a848e26` (same client) — 98,304 impressions, position 4.9, CTR 0.24%. Confidence: solid.

19. `content_963de14b1f58978f` (client `...e547b89c05043229`) — 97,312 impressions, position 3.8, CTR 0.50%. Confidence: moderate.

20. `content_cd3d932d4e1c8db0` (client `...9958f0a7ae1df715`) — 89,332 impressions, position 7.8, CTR 0.0045% — very thin again, though less extreme than #9/#11. Confidence: low-moderate; worth a manual click-count check.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Confirm the near-zero-click anomaly (rows 9, 11, 20)
suspect_ids = ['content_8e1334d6356668e3', 'content_fec55986a1868d62', 'content_cd3d932d4e1c8db0']
df[df['content_hash_id'].isin(suspect_ids)][
    ['content_hash_id', 'impressions_month', 'clicks_month', 'ctr']
]

# 2. Confirm client concentration in the top 20
print(queue.head(20)['client_hash_id'].value_counts())

# 3. Leakage check — confirm no product-decision fields in schema, and no future dates
print(con.sql(f"DESCRIBE SELECT * FROM {DAILY} LIMIT 1").df()['column_name'].tolist())
print(con.sql(f"SELECT MIN(report_date), MAX(report_date) FROM {DAILY}").df())

client_hash_id
client_e547b89c05043229    10
client_73cda7b4e4f265ea     9
client_9958f0a7ae1df715     1
Name: count, dtype: int64
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  min(report_date) max(report_date)
0       2026-03-01       2026-03-31


Weak picks — two real problems, not one:

Rows 9 and 11 (CTR ≈ 0.0007–0.0008%) are almost certainly noise or a tracking issue, not a genuine "low CTR" SEO problem. At 130K+ impressions, a CTR that low implies roughly 1 click total for the month — that's either a broken/miscounted click event, a canonical-tag issue routing clicks elsewhere, or a genuine data artifact. I wouldn't send a content writer to "improve the title" here without checking raw click counts first — extremely low-click, high-impression pages need a data-integrity check before an editorial one.
Structural weak point in the rule itself: client concentration. All 20 top rows come from just 3 clients (...e547b89c05043229, ...73cda7b4e4f265ea, ...9958f0a7ae1df715). Because score = impressions_month with no per-client normalization, the biggest/highest-traffic clients mechanically dominate every top-K slot — smaller clients with a genuinely severe CTR gap (but lower absolute impressions) never surface. This means the queue is really answering "which pages have the most raw traffic among stale ones," not "which pages most need review" — a real limitation worth fixing before Week 5's model, e.g. by ranking within-client or using a normalized/log-scaled score instead of raw impressions.

Leakage check: No product-decision fields (health_score, priority_score, action_type) present in this schema — confirmed via DESCRIBE in Week 3. All inputs come from March 2026 only, the same month the score describes — nothing from April onward. trend_pct/trend_direction don't exist in this warehouse table at all, so nothing label-derived leaked in.

Note on precision@K: Lane 3 has no observed outcome label, so precision@K (which needs ground-truth labels) doesn't apply here. The top-20 hand review above is the honest substitute — and it did its job: it surfaced a real weakness (client-volume dominance) that a clean-looking score alone would have hidden.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.